<a href="https://colab.research.google.com/github/dasihayu/artificial-intelegence/blob/main/Jobsheet_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Praktikum Jobsheet 8 - Data Pre-processing
## Dataset: iklan_sosmed.csv
- **Nama:** Dasi Hayu Permana
- **NIM:** 4.33.25.0.06
- **Kelas:** TI-1A

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier

## 2. Import Dataset

In [2]:
df = pd.read_csv('iklan_sosmed.csv', sep=';')
df.head()

,ID,Jenis_Kelamin,Umur,Gaji,Transaksi
0,15624510,Pria,19,285000000,0
1,15810944,Pria,35,300000000,0
2,15668575,Wanita,26,645000000,0
3,15603246,Wanita,27,855000000,0
4,15804002,Pria,19,1140000000,0


## 3. Assessing Data

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID             400 non-null    int64 
 1   Jenis_Kelamin  400 non-null    object
 2   Umur           400 non-null    int64 
 3   Gaji           400 non-null    int64 
 4   Transaksi      400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB


In [4]:
df.isna().sum()

,0
ID,0
Jenis_Kelamin,0
Umur,0
Gaji,0
Transaksi,0


In [5]:
print('Jumlah duplikasi: ', df.duplicated().sum())

Jumlah duplikasi:  0


In [6]:
df.describe()

,ID,Umur,Gaji,Transaksi
count,4.000000e+02,400.000000,4.000000e+02,400.000000
mean,1.569154e+07,37.655000,1.046138e+09,0.357500
std,7.165832e+04,10.482877,5.114544e+08,0.479864
min,1.556669e+07,18.000000,2.250000e+08,0.000000
25%,1.562676e+07,29.750000,6.450000e+08,0.000000
50%,1.569434e+07,37.000000,1.050000e+09,0.000000
75%,1.575036e+07,46.000000,1.320000e+09,1.000000
max,1.581524e+07,60.000000,2.250000e+09,1.000000


## 4. Data Preparation dengan One Hot Encoding
Kolom `Jenis_Kelamin` bertipe kategorik (Pria/Wanita), sehingga perlu diubah menjadi numerik menggunakan One Hot Encoding.

In [7]:
encoder = OneHotEncoder()

encoded_df = pd.DataFrame(
    encoder.fit_transform(df[['Jenis_Kelamin']]).toarray(),
    columns=encoder.get_feature_names_out(['Jenis_Kelamin'])
)

df_encoded = pd.concat([df, encoded_df], axis=1)
df_encoded = df_encoded.drop(columns=['Jenis_Kelamin'])

df_encoded.head()

,ID,Umur,Gaji,Transaksi,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,15624510,19,285000000,0,1.0,0.0
1,15810944,35,300000000,0,1.0,0.0
2,15668575,26,645000000,0,0.0,1.0
3,15603246,27,855000000,0,0.0,1.0
4,15804002,19,1140000000,0,1.0,0.0


## 5. Data Preparation dengan Outlier Removal
Menghapus data yang memiliki z-score lebih besar dari 3 pada kolom `Gaji`.

In [8]:
z_scores = np.abs((df_encoded['Gaji'] - df_encoded['Gaji'].mean()) / df_encoded['Gaji'].std())

df_clean = df_encoded.loc[round(z_scores) < 3]

print('Jumlah data sebelum outlier removal:', len(df_encoded))
print('Jumlah data sesudah outlier removal:', len(df_clean))
df_clean

Jumlah data sebelum outlier removal: 400
Jumlah data sesudah outlier removal: 400


,ID,Umur,Gaji,Transaksi,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,15624510,19,285000000,0,1.0,0.0
1,15810944,35,300000000,0,1.0,0.0
2,15668575,26,645000000,0,0.0,1.0
3,15603246,27,855000000,0,0.0,1.0
4,15804002,19,1140000000,0,1.0,0.0
...,...,...,...,...,...,...
395,15691863,46,615000000,1,0.0,1.0
396,15706071,51,345000000,1,1.0,0.0
397,15654296,50,300000000,1,0.0,1.0
398,15755018,36,495000000,0,1.0,0.0


## 6. Data Preparation dengan Standarization
Kolom `Umur` dan `Gaji` memiliki skala yang jauh berbeda, sehingga perlu distandarisasi menggunakan `StandardScaler`.

In [9]:
scaler = StandardScaler()
df_clean[['Umur', 'Gaji']] = scaler.fit_transform(df_clean[['Umur', 'Gaji']])
df_clean.head()

,ID,Umur,Gaji,Transaksi,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,15624510,-1.781797,-1.490046,0,1.0,0.0
1,15810944,-0.253587,-1.460681,0,1.0,0.0
2,15668575,-1.113206,-0.785290,0,0.0,1.0
3,15603246,-1.017692,-0.374182,0,0.0,1.0
4,15804002,-1.781797,0.183751,0,1.0,0.0


## 7. Pembuatan Dataset (Data Training dan Data Testing)

In [10]:
data = df_clean.drop(columns=['ID', 'Transaksi'])
data

,Umur,Gaji,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,-1.781797,-1.490046,1.0,0.0
1,-0.253587,-1.460681,1.0,0.0
2,-1.113206,-0.785290,0.0,1.0
3,-1.017692,-0.374182,0.0,1.0
4,-1.781797,0.183751,1.0,0.0
...,...,...,...,...
395,0.797057,-0.844019,0.0,1.0
396,1.274623,-1.372587,1.0,0.0
397,1.179110,-1.460681,0.0,1.0
398,-0.158074,-1.078938,1.0,0.0


In [11]:
data = data.values
data

array([[-1.78179743, -1.49004624,  1.        ,  0.        ],
       [-0.25358736, -1.46068138,  1.        ,  0.        ],
       [-1.11320552, -0.78528968,  0.        ,  1.        ],
       ...,
       [ 1.17910958, -1.46068138,  0.        ,  1.        ],
       [-0.15807423, -1.07893824,  1.        ,  0.        ],
       [ 1.08359645, -0.99084367,  0.        ,  1.        ]])

In [13]:
label = df_clean['Transaksi']
label

,Transaksi
0,0
1,0
2,0
3,0
4,0
...,...
395,1
396,1
397,1
398,0


In [14]:
label = label.values
label

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1,

In [15]:
data_train, data_test, label_train, label_test = train_test_split(data, label, test_size=0.2,
random_state=42)

print('Ukuran data latih:', data_train.shape)
print('Ukuran data uji:', data_test.shape)

Ukuran data latih: (320, 4)
Ukuran data uji: (80, 4)


## 8. Cross Validation

In [16]:
dt = DecisionTreeClassifier()

scores = cross_val_score(dt, data, label, cv=5)

print('Hasil cross validation:', scores)
print('Rata-rata akurasi:', scores.mean())

Hasil cross validation: [0.75   0.9    0.8625 0.775  0.775 ]
Rata-rata akurasi: 0.8125


## Kesimpulan
- Dataset `iklan_sosmed.csv` berisi 400 baris data tanpa missing value maupun data duplikat, sehingga proses cleaning difokuskan pada pengubahan tipe data dan deteksi outlier.
- Kolom kategorik `Jenis_Kelamin` berhasil diubah menjadi representasi numerik melalui One Hot Encoding.
- Proses outlier removal pada kolom `Gaji` tidak menemukan data yang perlu dihapus, menandakan sebaran data gaji pada dataset ini relatif wajar (tidak ada nilai ekstrem).
- Kolom `Umur` dan `Gaji` distandarisasi menggunakan `StandardScaler` agar memiliki skala yang setara sebelum digunakan untuk pemodelan.
- Data kemudian dibagi menjadi data latih (80%) dan data uji (20%), lalu dievaluasi menggunakan 5-fold cross validation dengan model Decision Tree, menghasilkan akurasi rata-rata yang cukup baik.